# Team pressure analysis -- LLM narrative report
Reads `pressure_analysis_llm_default.json` (built by Section 21 of `team_pressure_analysis_v30.ipynb`) and sends it to Claude to produce a full detailed tactical write-up.

**Only the default payload is ever used here** -- meta (methodology + data-quality caveats) + the 20 team-level aggregate tables + `notable_events`. The full export with 1,011 raw event/pass rows is never loaded or sent: this notebook has no code path that touches `pressure_analysis_llm_export.json` at all, by design.

Order: setup -> pricing table -> system prompt -> **cheap smoke test** (Haiku, ~$0.01) -> the real report (Opus, a few cents) -> save -> actual cost from the real response.


## Setup

In [ ]:
import json
import os
import anthropic
import paths

LLM_EXPORT_DEFAULT_PATH = getattr(paths, "LLM_EXPORT_DEFAULT_PATH", "pressure_analysis_llm_default.json")

with open(LLM_EXPORT_DEFAULT_PATH) as f:
    payload = json.load(f)

# sanity check: this must never be the full export -- fail loudly rather than silently
# sending raw event rows if someone points this at the wrong file later
assert set(payload.keys()) == {"meta", "team", "notable_events"}, (
    "payload has unexpected top-level keys -- make sure this is the DEFAULT export, "
    "not the full one with raw event/pass rows"
)

team_names = list(payload["meta"]["team_names"].values())
print(f"loaded default payload for {team_names[0]} vs {team_names[1]}")
print("meta keys:", list(payload["meta"].keys()))
print("team keys:", list(payload["team"].keys()))
print("notable_events keys:", list(payload["notable_events"].keys()))

# API key resolution, in order: real environment variable -> .env file (python-dotenv, if
# installed) -> paths.py fallback. Never hardcode the key itself in this notebook.
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()  # looks for a .env file in the current/parent directories
        api_key = os.environ.get("ANTHROPIC_API_KEY")
    except ImportError:
        pass
if not api_key:
    api_key = getattr(paths, "ANTHROPIC_API_KEY", None)

if not api_key:
    raise RuntimeError(
        "No ANTHROPIC_API_KEY found. Set it as an environment variable before starting Jupyter "
        "(PowerShell: $env:ANTHROPIC_API_KEY = \"sk-ant-...\"), or put ANTHROPIC_API_KEY=sk-ant-... "
        "in a .env file next to paths.py (pip install python-dotenv, and add .env to .gitignore), "
        "or add an ANTHROPIC_API_KEY constant to paths.py."
    )

client = anthropic.Anthropic(api_key=api_key)

# pricing as of the last time this was checked against
# https://platform.claude.com/docs/en/about-claude/pricing -- $ per million tokens (input, output)
PRICING_PER_MTOK = {
    "claude-haiku-4-5-20251001": (1, 5),
    "claude-sonnet-5": (2, 10),
    "claude-opus-5": (5, 25),
}

def _cost(model, usage):
    in_rate, out_rate = PRICING_PER_MTOK.get(model, (None, None))
    if in_rate is None:
        return None
    return usage.input_tokens / 1_000_000 * in_rate + usage.output_tokens / 1_000_000 * out_rate


## System prompt
Sets the role, tells the model exactly what it has (and doesn't have), and forces it to synthesize across categories rather than restate numbers table by table. Reused for both the smoke test and the real call, so the smoke test is an honest check of the real prompt.


In [ ]:
SYSTEM_PROMPT = """You are a professional football (soccer) tactical performance analyst writing a \
post-match pressing report for a coaching staff.

You will receive one JSON payload with three parts:
- meta: pitch/match constants, a `methodology` block defining exactly what every metric means \
(thresholds used, zone conventions, how possession/pressure events are segmented), and a \
`data_quality` block with the pipeline's known caveats (ball-carrier coverage, player-identity \
fragmentation).
- team: ~20 aggregate tables covering pressing volume, PPDA, zone breakdown, defensive line \
height, team shape/compactness, counter-press response, pass outcomes under pressure, reception \
triggers, and a pressure heatmap grid.
- notable_events: a small curated set of specific match moments (longest press per team, \
fastest/slowest counter-presses, most heavily pressured possessions) with timestamps.

You do NOT have access to frame-by-frame or per-event raw data beyond what's in notable_events -- \
do not claim to have looked at "every pass" or "every possession"; ground claims only in the \
aggregates and named events actually given to you.

Write a detailed tactical report, not a numbers readout. For every section: state the number, \
then explain what it means tactically, using the methodology block to interpret it correctly \
(e.g. what actually counts as "pressure" here). Explicitly cross-reference metrics across \
categories -- e.g. tie PPDA together with regain rate and defensive line height to characterize \
a team's pressing philosophy, rather than reporting each in isolation. Cite specific \
notable_events as concrete evidence for claims. Compare the two teams directly rather than \
describing them in separate, parallel sections. Where the data_quality caveats materially affect \
confidence in a conclusion, say so briefly rather than ignoring them or over-qualifying \
everything."""

REPORT_STRUCTURE = """Structure the report with clear section headers covering: pressing identity & \
philosophy per team, zone-by-zone pressing effectiveness, defensive shape & line height, \
counter-pressing effectiveness, passes under pressure & disruption, and a final comparative \
verdict on which team's pressing was more effective and why."""

print(f"system prompt: {len(SYSTEM_PROMPT)} chars")


## 0. Free dry run -- no API call, $0
Builds the exact same request the Haiku smoke test and the real Opus call will send -- same `SYSTEM_PROMPT`, same payload serialization -- but never sends it to the API. This confirms the payload loads correctly, `json.dumps` round-trips cleanly, and shows you the real size/token estimate, for genuinely zero cost, before you spend even the ~1 cent the smoke test below costs.


In [ ]:
DRY_RUN_MESSAGE = (
    "Here is the match pressure-analysis payload (JSON). In exactly 3 sentences: (1) confirm "
    "which two teams and what match this is, (2) state one specific number from the team-level "
    "data, (3) give one concrete tactical observation that number implies. No generic remarks.\n\n"
    + json.dumps(payload, separators=(",", ":"))
)

dry_run_bytes = len(DRY_RUN_MESSAGE.encode("utf-8")) + len(SYSTEM_PROMPT.encode("utf-8"))
dry_run_tokens_est = dry_run_bytes / 4  # same rough rule of thumb as Section 21's cost check

print("DRY RUN -- nothing sent to the API, $0 spent")
print(f"payload teams: {team_names[0]} vs {team_names[1]}")
print(f"request size: {dry_run_bytes:,} bytes (~{dry_run_tokens_est:,.0f} tokens estimated)")
print()
print("first 300 chars of what would be sent:")
print(DRY_RUN_MESSAGE[:300] + "...")
print()
print("if this looks right (correct team names, valid JSON, no truncation/errors above),")
print("run the Haiku smoke test cell below next -- it costs about a cent.")


## 1. Cheap smoke test before the real send
Runs the exact same system prompt and the exact same payload through the **cheapest model (Haiku 4.5)** with a tiny output cap -- just to confirm the API key works, the payload round-trips through `json.dumps` cleanly, and the model can actually parse real numbers out of it and say something concrete (not a generic non-answer). This costs a fraction of a cent, so run it before spending on the full Opus report below.

If the test response is vague, refuses, or clearly isn't grounded in the actual numbers, stop and fix the payload/prompt first -- don't proceed to the expensive call.


In [ ]:
TEST_MODEL = "claude-haiku-4-5-20251001"
TEST_MAX_TOKENS = 300

test_user_message = DRY_RUN_MESSAGE  # built for free in the dry-run cell above

# pre-flight cost estimate + confirmation -- nothing is sent until you type y
_est_in_tokens = dry_run_tokens_est  # from the Section 0 dry run above
_in_rate, _out_rate = PRICING_PER_MTOK.get(TEST_MODEL, (None, None))
if _in_rate is not None:
    _est_cost = _est_in_tokens / 1_000_000 * _in_rate + TEST_MAX_TOKENS / 1_000_000 * _out_rate
    print(f"About to call {TEST_MODEL}")
    print(f"  estimated input:  ~{_est_in_tokens:,.0f} tokens")
    print(f"  max output:       {TEST_MAX_TOKENS} tokens")
    print(f"  estimated cost:   ~${_est_cost:.4f}  (worst case, if it uses the full output cap)")
else:
    print(f"About to call {TEST_MODEL} -- no pricing on file, cost unknown")

if input("Proceed with this API call? [y/N]: ").strip().lower() != "y":
    raise SystemExit("Cancelled -- no API call made, $0 spent.")

test_response = client.messages.create(
    model=TEST_MODEL,
    max_tokens=TEST_MAX_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": test_user_message}],
)

test_text = "".join(block.text for block in test_response.content if block.type == "text")
print("TEST RESPONSE:")
print(test_text)
print()

test_cost = _cost(TEST_MODEL, test_response.usage)
print(f"actual tokens: input {test_response.usage.input_tokens:,} / output {test_response.usage.output_tokens:,}")
print(f"actual cost:   ~${test_cost:.4f}" if test_cost is not None else "actual cost: (model not in PRICING_PER_MTOK)")


## The real report
Only run this once the smoke test above looks grounded and correct. Model choice: Opus 5 by default -- at this payload size (a few cents per call, per Section 21's cost check) the quality difference matters more than the marginal cost. Swap `MODEL` to `"claude-sonnet-5"` or `"claude-haiku-4-5-20251001"` to cut cost further while iterating on the prompt itself.


In [ ]:
MODEL = "claude-opus-5"
MAX_OUTPUT_TOKENS = 8000  # a genuinely detailed multi-section report needs headroom

user_message = (
    "Here is the match pressure-analysis payload (JSON). Write the full detailed tactical "
    "report as instructed.\n\n" + REPORT_STRUCTURE + "\n\n" + json.dumps(payload, separators=(",", ":"))
)

# pre-flight cost estimate + confirmation -- nothing is sent until you type y
_est_in_tokens = (len(user_message.encode("utf-8")) + len(SYSTEM_PROMPT.encode("utf-8"))) / 4
_in_rate, _out_rate = PRICING_PER_MTOK.get(MODEL, (None, None))
if _in_rate is not None:
    _est_cost = _est_in_tokens / 1_000_000 * _in_rate + MAX_OUTPUT_TOKENS / 1_000_000 * _out_rate
    print(f"About to call {MODEL}")
    print(f"  estimated input:  ~{_est_in_tokens:,.0f} tokens")
    print(f"  max output:       {MAX_OUTPUT_TOKENS:,} tokens")
    print(f"  estimated cost:   ~${_est_cost:.4f}  (worst case, if it uses the full output cap --")
    print(f"                     actual is usually well under this since reports rarely hit the cap)")
else:
    print(f"About to call {MODEL} -- no pricing on file, cost unknown")

if input("Proceed with this API call? [y/N]: ").strip().lower() != "y":
    raise SystemExit("Cancelled -- no API call made, $0 spent.")

response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_OUTPUT_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": user_message}],
)

report_text = "".join(block.text for block in response.content if block.type == "text")
print(f"received {len(report_text)} chars, stop_reason={response.stop_reason}")


## Save the report

In [ ]:
REPORT_PATH = getattr(paths, "LLM_REPORT_PATH", "pressure_analysis_report.md")

with open(REPORT_PATH, "w") as f:
    f.write(report_text)

print(f"wrote {REPORT_PATH}")
print()
print(report_text[:1500] + ("..." if len(report_text) > 1500 else ""))


## Actual cost (from the real API response, not an estimate)

In [ ]:
real_cost = _cost(MODEL, response.usage)
print(f"input_tokens:  {response.usage.input_tokens:,}")
print(f"output_tokens: {response.usage.output_tokens:,}")
print(f"real call cost: ~${real_cost:.4f}" if real_cost is not None else f"(add {MODEL!r} to PRICING_PER_MTOK)")
print(f"smoke test cost was: ~${test_cost:.4f}")
print(f"total this notebook run: ~${(real_cost or 0) + (test_cost or 0):.4f}")
